# 01 EDA

Kesifsel veri analizi (EDA) ve rapor icin temel veri kanitlari.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.fairness import initial_target_distribution_by_group
from src.preprocessing import (
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    validate_test_data,
    validate_train_test_compatibility,
    validate_training_data,
)
from src.project_paths import INCOME_CSV, INCOME_TEST_CSV, OUTPUTS_DIR, FIGURES_DIR, REPORTS_DIR

sns.set_theme(style='whitegrid')
OUTPUTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

In [ ]:
train_df = pd.read_csv(INCOME_CSV)
test_df = pd.read_csv(INCOME_TEST_CSV)

validate_training_data(train_df)
validate_test_data(test_df)
validate_train_test_compatibility(train_df, test_df)

print(f'train shape: {train_df.shape}')
print(f'test shape: {test_df.shape}')

In [ ]:
class_distribution = (
    train_df['income']
    .value_counts(dropna=False)
    .rename_axis('income')
    .reset_index(name='count')
)
class_distribution['rate'] = class_distribution['count'] / len(train_df)
class_distribution.to_csv(OUTPUTS_DIR / 'eda_class_distribution.csv', index=False)
class_distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=class_distribution, x='income', y='count', ax=ax)
ax.set_title('Income class distribution')
ax.set_xlabel('Income')
ax.set_ylabel('Count')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'class_distribution.png', dpi=160)
plt.show()

In [ ]:
missing_summary = pd.DataFrame({
    'column': train_df.columns,
    'train_missing_rate': train_df.isna().mean().values,
    'test_missing_rate': [test_df[col].isna().mean() if col in test_df.columns else None for col in train_df.columns],
})
missing_summary.to_csv(OUTPUTS_DIR / 'eda_missing_values.csv', index=False)
missing_summary.sort_values('train_missing_rate', ascending=False)

In [ ]:
plot_missing = missing_summary[missing_summary['column'] != 'income'].copy()
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=plot_missing, x='column', y='train_missing_rate', ax=ax, color='#4C78A8')
ax.set_title('Training missing value rate')
ax.set_xlabel('Column')
ax.set_ylabel('Missing rate')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'missing_values.png', dpi=160)
plt.show()

In [ ]:
numeric_summary = train_df[list(NUMERIC_FEATURES)].describe().T
categorical_summary = train_df[list(CATEGORICAL_FEATURES)].nunique(dropna=False).rename('unique_values').reset_index()
categorical_summary.columns = ['column', 'unique_values']
numeric_summary.to_csv(OUTPUTS_DIR / 'eda_numeric_summary.csv')
categorical_summary.to_csv(OUTPUTS_DIR / 'eda_categorical_summary.csv', index=False)
display(numeric_summary)
display(categorical_summary)

In [ ]:
gender_income_distribution = initial_target_distribution_by_group(train_df)
gender_income_distribution.to_csv(OUTPUTS_DIR / 'eda_gender_income_distribution.csv', index=False)
gender_income_distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=gender_income_distribution, x='sex', y='rate', hue='income', ax=ax)
ax.set_title('Income distribution by sex')
ax.set_xlabel('Sex')
ax.set_ylabel('Rate')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'gender_income_distribution.png', dpi=160)
plt.show()

In [ ]:
high_rates = (
    gender_income_distribution[gender_income_distribution['income'] == 'high']
    .set_index('sex')['rate']
    .to_dict()
)
notes = f'''# Rolling Report Notes

## T4 - EDA bulgulari

- Train verisi {train_df.shape[0]} satir ve {train_df.shape[1]} kolondan olusuyor; test verisi {test_df.shape[0]} satir ve {test_df.shape[1]} kolondan olusuyor.
- Hedef sinif dagilimi: low {class_distribution.loc[class_distribution['income'] == 'low', 'rate'].iloc[0]:.3f}, high {class_distribution.loc[class_distribution['income'] == 'high', 'rate'].iloc[0]:.3f}.
- En yuksek eksik deger oranlari `ability to speak english` ve `gave birth this year` kolonlarinda goruldu; bu T7 ablation deneyi icin kanit saglar.
- `sex` bazinda high income orani Female icin {high_rates.get('Female', float('nan')):.3f}, Male icin {high_rates.get('Male', float('nan')):.3f}; bu fairness (gender fairness) tartismasi icin baslangic sinyalidir.

'''
(REPORTS_DIR / 'report_notes_tr.md').write_text(notes, encoding='utf-8')
print(notes)